In [41]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from ete3 import Tree

path_dataset = Path("datasets")
path_coi = path_dataset / "COI_DB"
path_fishtree = path_dataset / "fishtree"
path_coi_taxonomy = path_coi / "CO1DB_taxanomy.txt"
path_coi_fasta = path_coi / "COI90.fasta"
path_fishtree_taxonomy = path_fishtree / "PFC_taxonomy.csv"
path_fishtree_tree = path_fishtree / "actinopt_12k_raxml.tre"

In [42]:
names = ['id', 'domain', 'kingdom', 'phylum', 'sC', 'superclass', 'order', 'family', 'genus', 'genus.species']
usecols = [n for n in names if n not in ['domain', 'kingdom', 'phylum', 'sC']]
coi_taxonomy = pd.read_csv(path_coi_taxonomy, sep='(?:\t|;)(?:D|K|P|sC|C|O|F|G|s)_', engine='python', header=None, index_col=0,
                           names=names, usecols=usecols)
coi_taxonomy

,superclass,order,family,genus,genus.species
id,,,,,
COI_211_FI00000726,Actinopterygii,Spariformes,Sparidae,Diplodus,Diplodus sargus ascensionis
COI_215_FI00000727,Actinopterygii,Spariformes,Sparidae,Diplodus,Diplodus sargus ascensionis
COI_319_FI00000731,Actinopterygii,Spariformes,Sparidae,Diplodus,Diplodus sargus lineatus
COI_512_FI00000728,Actinopterygii,Spariformes,Sparidae,Diplodus,Diplodus sargus helenae
COI_513_FI00000729,Actinopterygii,Spariformes,Sparidae,Diplodus,Diplodus sargus helenae
...,...,...,...,...,...
COI_ZSM-PIS-046847|ZSM-P-AA-1262_FI00000254,Actinopterygii,Gobiiformes,Eleotridae,Oxyeleotris,Oxyeleotris marmorata
COI_ZSM-PIS-047039|ZSM-P-AA-1277_FI00000479,Actinopterygii,Anabantiformes,Osphronemidae,Osphronemus,Osphronemus goramy
COI_ZSM-PIS-047039|ZSM-P-AA-1278_FI00000480,Actinopterygii,Anabantiformes,Osphronemidae,Osphronemus,Osphronemus goramy


In [43]:
coi_fasta = pd.read_csv(path_coi_fasta, lineterminator='>', header=None)
coi_fasta = coi_fasta[0].str.split('\n', expand=True, n=1).set_axis(["id", "sequence"], axis=1)
coi_fasta['id'] = coi_fasta['id'].str.strip()
coi_fasta['sequence'] = coi_fasta['sequence'].str.upper().str.replace('[^A-Z]', '', regex=True)
coi_fasta['seq_len'] = coi_fasta['sequence'].str.len()
coi_fasta['N_frac'] = coi_fasta['sequence'].str.count('N') / coi_fasta['seq_len']
coi_fasta

,id,sequence,seq_len,N_frac
0,COI_Gn00001_00000001,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0
1,COI_Gn00001_00000002,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0
2,COI_Gn00001_00000003,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0
3,COI_Gn00001_00000004,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0
4,COI_Gn00001_00000005,TGAGCTGGGATAGTAGGCACAGCTTTAAGCTTGCTAATCCGAGCAG...,618,0.0
...,...,...,...,...
153699,COI_Gn03825_00192547,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0
153700,COI_Gn03825_00192548,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0
153701,COI_Gn03825_00192549,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0
153702,COI_Gn03825_00192550,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0


In [44]:
coi_fasta_taxonomy = pd.merge(coi_fasta, coi_taxonomy, how='inner', on='id')
coi_fasta_taxonomy

,id,sequence,seq_len,N_frac,superclass,order,family,genus,genus.species
0,COI_Gn00001_00000001,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0,Actinopterygii,Tetraodontiformes,Balistidae,Abalistes,Abalistes stellatus
1,COI_Gn00001_00000002,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0,Actinopterygii,Tetraodontiformes,Balistidae,Abalistes,Abalistes stellatus
2,COI_Gn00001_00000003,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0,Actinopterygii,Tetraodontiformes,Balistidae,Abalistes,Abalistes stellatus
3,COI_Gn00001_00000004,GCACCCTTTACCTAATTTTTGGTGCTTGAGCTGGGATAGTAGGCAC...,663,0.0,Actinopterygii,Tetraodontiformes,Balistidae,Abalistes,Abalistes stellatus
4,COI_Gn00001_00000005,TGAGCTGGGATAGTAGGCACAGCTTTAAGCTTGCTAATCCGAGCAG...,618,0.0,Actinopterygii,Tetraodontiformes,Balistidae,Abalistes,Abalistes stellatus
...,...,...,...,...,...,...,...,...,...
153699,COI_Gn03825_00192547,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0,Actinopterygii,Siluriformes,Pimelodidae,Zungaro,Zungaro jahu
153700,COI_Gn03825_00192548,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0,Actinopterygii,Siluriformes,Pimelodidae,Zungaro,Zungaro jahu
153701,COI_Gn03825_00192549,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0,Actinopterygii,Siluriformes,Pimelodidae,Zungaro,Zungaro jahu
153702,COI_Gn03825_00192550,ATTCGGGCAGAGCTGGCCCAACCCGGCGCCCTCCTAGGTGATGACC...,546,0.0,Actinopterygii,Siluriformes,Pimelodidae,Zungaro,Zungaro jahu


In [45]:
t = Tree(str(path_fishtree_tree))
t_leaves = t.get_leaves()
df_leaves = pd.DataFrame([l.name.replace("_", " ") for l in t_leaves], columns=['genus.species'])
df_leaves['genus.species'] = df_leaves['genus.species'].str.replace(r'\b(\w+) \1\b', r'\1', regex=True)
df_leaves

,genus.species
0,Gambusia marshi
1,Gambusia panuco
2,Gambusia regani
3,Gambusia aurata
4,Gambusia hurtadoi
...,...
11633,Polypterus congicus
11634,Polypterus retropinnis
11635,Polypterus mokelembembe
11636,Polypterus ornatipinnis


In [46]:
df_taxonomy_leaves = pd.merge(coi_fasta_taxonomy, df_leaves, how='inner', on='genus.species')
df_taxonomy_leaves_count = df_taxonomy_leaves.value_counts(subset=['order', 'family', 'genus.species']).to_frame()
df_taxonomy_leaves_count.head(20)

count
order              family        genus.species                     
Perciformes        Sciaenidae    Larimichthys polyactis         573
                   Polynemidae   Eleutheronema tetradactylum    543
Cypriniformes      Cyprinidae    Cyprinus carpio                452
Gadiformes         Gadidae       Gadus morhua                   431
Clupeiformes       Clupeidae     Sardinella longiceps           411
Scombriformes      Scombridae    Thunnus albacares              403
                                 Scomber scombrus               399
Salmoniformes      Salmonidae    Oncorhynchus mykiss            392
Esociformes        Esocidae      Esox lucius                    361
Mugiliformes       Mugilidae     Mugil cephalus                 355
Caproiformes       Coryphaenidae Coryphaena hippurus            350
Cypriniformes      Cyprinidae    Notemigonus crysoleucas        343
Cichliformes       Cichlidae     Oreochromis niloticus          327
Anabantiformes     Channidae     Channa striata                 308
Clupeiformes       Clupeidae     Brevoortia tyrannus            305
Salmoniformes      Salmonidae    Oncorhynchus tshawytscha       300
Perciformes        Sillaginidae  Sillago sihama                 300
Cyprinodontiformes Poeciliidae   Poecilia mexicana              295
Cypriniformes      Cyprinidae    Phoxinus phoxinus              289
Gadiformes         Merlucciidae  Merluccius merluccius          284

In [47]:
df_taxonomy_leaves_length = df_taxonomy_leaves.groupby(by="order").agg({'seq_len': ['count', 'min', 'median', 'max', 'std']})
df_taxonomy_leaves_length.head(30)

seq_len                                
                     count  min median    max          std
order                                                     
Acanthomorphata        344  336  648.0  16748  2109.318013
Acanthuriformes        294  517  546.0  17272  1357.349456
Acipenseriformes       548  193  654.0  16790  6035.841097
Albuliformes            64  568  652.0    652    14.947917
Alepocephaliformes     115  520  652.0  18235  5004.745276
Amiiformes              14  222  655.0  16989  6763.927908
Anabantiformes        1797  226  652.0  17099  2620.236190
Anguilliformes        1859  213  648.0  18978  4425.322157
Atheriniformes        1469  369  651.0  16909  5538.549755
Aulopiformes           510  312  652.0  17663  2586.582375
Batrachoidiformes       42  314  652.0    673    99.015878
Beloniformes           688  266  651.0  16953  3602.062795
Beryciformes           195  255  652.0  16537  4227.082415
Blenniiformes          762  415  652.0  16496   813.171345
Caproiformes          4125  136  651.0  18008  2262.820426
Carcharhiniformes       60  218  603.0  16280  3942.569692
Centrarchiformes      1561  207  651.0  17336  4310.902435
Chaetodontiformes     1113  313  638.0  16772  1501.259545
Characiformes         4468  110  634.0  17998  1578.488858
Cichliformes          1893  140  652.0  16663  3160.622819
Clupeiformes          3228  129  639.0  17555  4491.319529
Cypriniformes        15221  136  652.0  17865  3934.624147
Cyprinodontiformes    2962   93  652.0  19527  2192.818656
Elopiformes            119  222  652.0  16713  3989.496005
Ephippiformes          246  450  652.0  23152  2840.836135
Esociformes            529  400  652.0  16695  1761.209774
Gadiformes            2873  108  648.0  17078  5113.112518
Galaxiiformes          152  515  600.0  16502  1817.810531
Gerreiformes           448  242  652.0  16839  1698.798035
Gobiiformes           4153  245  652.0  18562  3376.414797